# 第56课 · 反向传播（backpropagation）手推 — 链式法则（chain rule）逐层展开，梯度（gradient） = 局部梯度 × 上游梯度

**目标**：实现 `Value.backward()`，从输出节点出发，拓扑排序（topological sort）后逐节点调用 `_backward()`，把梯度沿计算图（computation graph）向后传播。

🔗 **Aurora 连接**：这正是 PyTorch 的 `.backward()` 在做的事情；Month 2 训练循环（权重更新）依赖此机制。

← **上一课**　[L55 · Value 算子补全](L55_forward_pass.ipynb)

> 上节课学习了 **Value 算子补全**：pow、relu、tanh、exp 节点实现，计算图完整展开。  
> 本课将探讨 **反向传播手推**。

## 本课剧情：顺序错了，梯度就错了

前两节课给每个算子装上了"局部反向函数"。但有个问题没解决：**按什么顺序调用这些函数？**

错误示范：随机顺序调用 `_backward()`。考虑计算图 `L = a*b + c`：
- 若先调 `a*b` 的 `_backward()`，此时 `(a*b).grad` 还是 0（`+` 还没传过来）→ `a.grad` 和 `b.grad` 算出来全是 0

**结论：必须从输出节点往输入节点逆向，且每个节点只在它的所有后续节点都处理完之后才能调用。** 这就是 DAG 的**逆拓扑序（reverse topological order）**。

算法只需三步：

```
1. 从 L 出发，DFS 后序遍历 DAG → 得到拓扑序列 topo
2. L.grad = 1.0                  （dL/dL = 1，链式法则的起点）
3. for node in reversed(topo):   → 逆序调用每个节点的 _backward()
       node._backward()
```

**梯度累积陷阱**：同一节点 `a` 被两个算子使用时（`L = a*a`），梯度从两条路径传来，必须 `+=` 累积，不能用 `=` 覆盖。

本节任务：实现 `Value.backward()`，让 `L.backward()` 一键完成整图反向传播。

In [ ]:
import numpy as np

# ── Value 基础实现（含各运算的 _backward，backward 留待下方实现）──
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _bwd():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _bwd
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _bwd():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _bwd
        return out

    def __neg__(self):        return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')
        def _bwd():
            self.grad += (out.data > 0) * out.grad
        out._backward = _bwd
        return out

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

print("Value 类加载完毕")

## 概念 1：拓扑排序保证梯度累积（gradient accumulation）顺序

计算图是一个 DAG（有向无环图）。节点 `v` 的 `_backward()` 需要用到 `v` 的所有"下游节点"（消费了 `v.data` 的节点）的梯度，这些梯度必须先准备好。

拓扑排序给出一个线性顺序，使得：若 `u → v`（`u` 依赖 `v`），则 `u` 排在 `v` 后面。
反向遍历此顺序，就能保证每个节点 `_backward()` 被调用时，其上游的累积梯度已经完整。

DFS 后序遍历（post-order）天然生成拓扑序：先递归访问所有子节点，访问完后再把当前节点加入列表。

```
topo = []
visited = set()
def build(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:   # 向输入方向走
            build(child)
        topo.append(v)          # 后序：最后加入自己
build(output)
# topo[-1] == output，topo[0] 是叶节点
```

In [ ]:
# 演示拓扑排序顺序
a = Value(2.0); b = Value(3.0); c = Value(-1.0)
L = a * b + c   # L = a*b + c

topo = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(L)
print("拓扑序（从叶到输出）：")
for i, v in enumerate(topo):
    print(f"  [{i}] op={v._op!r:5s}  data={v.data:.2f}  prev_count={len(v._prev)}")
print()
print("逆序（backward 的遍历方向）：", [v._op or 'leaf' for v in reversed(topo)])

## 概念 2：从输出节点出发，`self.grad = 1.0`

损失 `L` 对自身的梯度恒为 1：`dL/dL = 1`。
这是链式法则的起点——所有后续的 `_backward()` 都在把"上游梯度"（即 `out.grad`）乘以局部导数后写入输入节点的 `.grad`。

```
self.grad = 1.0          # dL/dL = 1
for node in reversed(topo):
    node._backward()     # 链式传播
```

如果忘记设置 `self.grad = 1.0`，所有乘以 `out.grad` 的项都变成 0，梯度全部为零。

In [ ]:
# 演示：不设 grad=1.0 时梯度全零
a2 = Value(2.0); b2 = Value(3.0)
out2 = a2 * b2
# 手动调用 _backward，但不设置 out2.grad
out2._backward()
print("未设 out.grad=1.0 时：a2.grad =", a2.grad, "（预期 3.0，但得到 0）")

# 正确做法
a3 = Value(2.0); b3 = Value(3.0)
out3 = a3 * b3
out3.grad = 1.0          # ← 关键
out3._backward()
print("设置 out.grad=1.0 后：a3.grad =", a3.grad, "（预期 3.0）✅")

## 概念 3：梯度累积（`+=`）而非覆盖（`=`）

当同一个节点被多个后续节点使用时（菱形图），梯度来自多条路径，必须**相加**，不能用后一条路径覆盖前一条。

例：`L = a*a`（`a` 同时作为左右操作数）
- 路径1：`dL/d(左a) = a.data = 2`
- 路径2：`dL/d(右a) = a.data = 2`
- 总梯度：`a.grad = 2 + 2 = 4`（等于解析式 `dL/da = 2a = 4`）

在 `_backward` 里统一写 `self.grad += ...` 而非 `self.grad = ...`，自动处理菱形图。

In [ ]:
# 菱形图演示：L = a * a
a4 = Value(2.0)
L4 = a4 * a4          # a4 同时是 _prev 里的两个"子节点"（同一对象）
L4.grad = 1.0
L4._backward()         # 执行一次 mul 的 _backward
print(f"a4.grad = {a4.grad:.1f}  （预期 4.0，因为 d(a^2)/da = 2a = 4）")
# 注意：mul 的 _backward 用 +=，所以两条路径都累积进来了

## 1. ✏️ 实现 `Value.backward(self)`

**三步实现路线**：

| 步骤 | 动作 | 关键点 |
|---|---|---|
| 1 | DFS 后序遍历 `self` → `topo` 列表 | 用 `visited` set 避免重复；遍历 `node._prev` |
| 2 | `self.grad = 1.0` | dL/dL=1，是所有梯度计算的起点 |
| 3 | `for node in reversed(topo): node._backward()` | 逆序保证"上游先算完" |

**验证标准**：
```python
a, b, c = Value(2.0), Value(3.0), Value(-1.0)
L = a * b + c    # L.data = 5.0
L.backward()
assert abs(a.grad - 3.0) < 1e-12   # dL/da = b = 3
assert abs(b.grad - 2.0) < 1e-12   # dL/db = a = 2
assert abs(c.grad - 1.0) < 1e-12   # dL/dc = 1
```

**菱形图验证**（`L = a*a`，`a=2`）：
- `dL/da` 来自两条路径：左乘(`a.data=2`)＋右乘(`a.data=2`) → `grad = 4.0`
- 若误用 `=` 而非 `+=`，只保留最后一条路径，得到 `2.0`（错误）

In [ ]:
def backward(self):
    # ✏️ TODO 步骤1：DFS 后序遍历，建立拓扑序列 topo
    topo = []
    visited = set()
    def build(v):
        raise NotImplementedError("TODO: 递归访问 v._prev，后序加入 topo")

    build(self)

    # ✏️ TODO 步骤2：设置输出节点梯度为 1.0
    # TODO: self.grad = 1.0

    # ✏️ TODO 步骤3：逆序调用每个节点的 _backward()
    pass

# 将方法绑定到 Value 类
Value.backward = backward  # 直接替换

In [ ]:
# ⚠️ 请在完成 Cell 10 的 backward() 实现后再运行本格。未实现时会捕获提示而非崩溃。
try:
    # 检查：L = a*b + c
    a = Value(2.0); b = Value(3.0); c = Value(-1.0)
    L = a * b + c
    L.backward()
    
    assert abs(a.grad - 3.0) < 1e-9, f"a.grad 应为 3.0，得到 {a.grad}"
    assert abs(b.grad - 2.0) < 1e-9, f"b.grad 应为 2.0，得到 {b.grad}"
    assert abs(c.grad - 1.0) < 1e-9, f"c.grad 应为 1.0，得到 {c.grad}"
    print(f"a.grad={a.grad:.1f}  b.grad={b.grad:.1f}  c.grad={c.grad:.1f}")
    print("✅ backward() 基本检查通过")
    
    # 菱形图检查：L = a*a，dL/da = 2*a = 4
    a2 = Value(2.0)
    L2 = a2 * a2
    L2.backward()
    assert abs(a2.grad - 4.0) < 1e-9, f"a2.grad 应为 4.0，得到 {a2.grad}"
    print(f"菱形图 a2.grad={a2.grad:.1f}（预期 4.0）")
    print("✅ 菱形图梯度累积检查通过")
except NotImplementedError:
    print('⚠️ TODO：请先在 Cell 10 中实现 backward()，再运行本格验证。')
except AssertionError as e:
    print(f'❌ 验证失败：{e}')
    print('💡 检查你的 topo 排序方向和 grad 累积逻辑。')


## 参数实验：数值微分（numerical differentiation）验证梯度

用数值微分 `(f(x+h) - f(x-h)) / (2h)` 独立计算梯度，与 `backward()` 结果对比，差距应 `< 1e-4`。

实验参数：
- `h = 1e-5`（步长太大误差大，太小浮点抖动）
- 测试函数：`L = a*b + c.relu()`（含 relu 的混合表达式）
- 预期现象：`a.grad` 等于 `b.data`，数值微分与解析梯度差距 `< 1e-4`

In [ ]:
def numeric_grad(f, x, h=1e-5):
    """对标量输入 x 计算 f 的数值梯度（中心差分）。"""""
    return (f(x + h) - f(x - h)) / (2 * h)

# 固定其他参数，逐一扰动输入
a_val, b_val, c_val = 2.0, 3.0, -0.5

def L_a(av): return (Value(av) * Value(b_val) + Value(c_val).relu()).data
def L_b(bv): return (Value(a_val) * Value(bv) + Value(c_val).relu()).data
def L_c(cv): return (Value(a_val) * Value(b_val) + Value(cv).relu()).data

num_a = numeric_grad(L_a, a_val)
num_b = numeric_grad(L_b, b_val)
num_c = numeric_grad(L_c, c_val)

# 解析梯度（backward）
a = Value(a_val); b = Value(b_val); c = Value(c_val)
L = a * b + c.relu()
L.backward()

print(f"{'变量':<6} {'backward':>12} {'数值微分':>12} {'误差':>12}")
for name, ana, num in [('a', a.grad, num_a), ('b', b.grad, num_b), ('c', c.grad, num_c)]:
    err = abs(ana - num)
    flag = "✅" if err < 1e-4 else "❌"
    print(f"{name:<6} {ana:>12.6f} {num:>12.6f} {err:>12.2e} {flag}")

## 本课收束

`Value.backward()` 通过 DFS 后序遍历建立拓扑序，再逆序调用每个节点的 `_backward()`，输出每个叶节点的 `.grad`（即损失对该参数的偏导（partial derivative）数）。
梯度用 `+=` 累积，保证菱形图（参数复用）下多路径贡献正确相加。
此机制直接对应后续训练循环的参数更新步骤（见 **L58** `L58_training_loop.ipynb`）：`w.data -= lr * w.grad`；Aurora 代码库 `src/aurora/` 中的训练脚本均遵循此模式。
下一节（L57）将在此基础上组装 `MLP`，把多层 `Value` 节点串联成真实网络，并跑一次完整的前向 + 反向 + 更新循环。

## ✏️ 闭卷推导检查格 — 反向传播矩阵公式

**规则：关闭上方所有格，先在此 Markdown 格写出推导，再在下方 Code 格实现。**

**题目**：2 层 MLP 前向传播：
$$z^{(1)} = W^{(1)} x + b^{(1)}, \quad a^{(1)} = \sigma(z^{(1)})$$
$$\hat{y} = W^{(2)} a^{(1)} + b^{(2)}, \quad L = \tfrac{1}{2}(\hat{y} - y)^2$$

推导（写在此处，不查笔记）：

1. $\dfrac{\partial L}{\partial W^{(2)}} = ?$（写成矩阵外积形式）

2. $\delta^{(1)} = \dfrac{\partial L}{\partial z^{(1)}} = ?$（含 $\sigma'(z^{(1)})$ 的 Hadamard 积）

3. $\dfrac{\partial L}{\partial W^{(1)}} = ?$（写成矩阵外积形式）

（在此处写推导...）

In [ ]:
# 验证：在下方 Code 格实现你推导的公式，与数值梯度对比
# 只需填写标注 TODO 的三行；其余代码已给出
import numpy as np

def sigmoid(z): return 1 / (1 + np.exp(-z))
def dsigmoid(z): s = sigmoid(z); return s * (1 - s)

np.random.seed(7)
W1 = np.random.randn(4, 3) * 0.1
b1 = np.zeros((4, 1))
W2 = np.random.randn(1, 4) * 0.1
b2 = np.zeros((1, 1))
x  = np.random.randn(3, 1)
y  = np.array([[1.0]])

# 前向传播
z1 = W1 @ x + b1; a1 = sigmoid(z1)
z2 = W2 @ a1 + b2; y_hat = z2
loss = 0.5 * (y_hat - y) ** 2

# ── 反向传播（按你的推导填写 TODO） ──────────────────────────────
dL_dyhat = y_hat - y
# TODO 1: dL/dW2  （shape: same as W2）
dL_dW2 = dL_dyhat @ a1.T          # ← 按你的推导修改

# TODO 2: δ¹ = dL/dz¹  （shape: same as z1）
delta1 = (W2.T @ dL_dyhat) * dsigmoid(z1)  # ← 按你的推导修改

# TODO 3: dL/dW1  （shape: same as W1）
dL_dW1 = delta1 @ x.T             # ← 按你的推导修改

# 数值梯度验证（有限差分）——参考值，不可改动
def loss_fn(W2_, W1_):
    a1_ = sigmoid(W1_ @ x + b1)
    yh  = W2_ @ a1_ + b2
    return 0.5 * float((yh - y) ** 2)

eps = 1e-5
def num_grad(param, i, j, loss_fn_partial):
    p = param.copy(); p[i,j] += eps
    return (loss_fn_partial(p) - loss_fn_partial(param.copy())) / eps

ng_W2_00 = num_grad(W2, 0, 0, lambda w: loss_fn(w, W1))
ng_W1_00 = num_grad(W1, 0, 0, lambda w: loss_fn(W2, w))

tol = 1e-4
ok_W2 = abs(dL_dW2[0,0] - ng_W2_00) < tol
ok_W1 = abs(dL_dW1[0,0] - ng_W1_00) < tol

print(f"dL/dW2[0,0]: 你的推导 = {dL_dW2[0,0]:+.6f}, 数值梯度 = {ng_W2_00:+.6f}  {'✅' if ok_W2 else '❌'}")
print(f"dL/dW1[0,0]: 你的推导 = {dL_dW1[0,0]:+.6f}, 数值梯度 = {ng_W1_00:+.6f}  {'✅' if ok_W1 else '❌'}")
if ok_W2 and ok_W1:
    print("\n✅ 反向传播推导验证通过")
else:
    print("\n❌ 公式有误，请对照推导重新检查 TODO 行")

## ✏️ 推导练习：手推两层网络的梯度

给定两层网络：`loss = ((W2 @ relu(W1 @ x)) - y)²`

**闭卷任务**：不看上方代码，在下方推导：
1. `∂loss/∂W2` = ？（提示：令 `h = relu(W1 @ x)`，`ŷ = W2 @ h`）
2. `∂loss/∂W1` = ？（提示：链式法则经过 relu）

在下方代码格中：用数值微分（有限差分法）验证你的推导是否正确。

In [ ]:
import numpy as np

# 简单两层网络（标量输出）
np.random.seed(0)
x  = np.array([1.0, 2.0])
y  = np.array([1.0])
W1 = np.random.randn(3, 2) * 0.5
W2 = np.random.randn(1, 3) * 0.5

def forward(W1, W2, x, y):
    h    = np.maximum(0, W1 @ x)   # relu
    yhat = W2 @ h
    loss = float(((yhat - y) ** 2).sum())
    return loss, h, yhat

loss, h, yhat = forward(W1, W2, x, y)

# ✏️ TODO: 在下方填写解析梯度（手推后填入）
raise NotImplementedError("TODO: 填入 dL_dW2 和 dL_dW1 的手推公式，再运行数值验证")

# --- 数值梯度验证（填完手推后再运行） ---
# eps = 1e-5
# num_grad_W2 = np.zeros_like(W2)
# for i in range(W2.shape[0]):
#     for j in range(W2.shape[1]):
#         W2p = W2.copy(); W2p[i,j] += eps
#         W2m = W2.copy(); W2m[i,j] -= eps
#         num_grad_W2[i,j] = (forward(W1,W2p,x,y)[0] - forward(W1,W2m,x,y)[0]) / (2*eps)
# np.testing.assert_allclose(dL_dW2, num_grad_W2, atol=1e-5)
# print("✅ dL/dW2 数值验证通过")

In [ ]:
# ✏️ 本课自评
l56_review = {
    "topo_order_understood":   None,  # 理解为什么必须逆拓扑序才能正确反向传播？True/False
    "backward_implemented":    None,  # backward() DFS+逆序+grad累积 全部正确实现？True/False
    "grad_init_understood":    None,  # 记住 L.grad=1.0 是链式法则起点，不是任意选的？True/False
    "accumulation_vs_assign":  None,  # 菱形图演示中 += 与 = 的区别验证通过？True/False
    "whiteboard_passed":       None,  # 白板手推反向传播矩阵公式推导通过？True/False
}

unfilled = [k for k, v in l56_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l56_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L56 全部通关！进入 L57：MLP 从零搭建')

---

→ **下一课**　[L57 · MLP 从零搭建](L57_mlp.ipynb)

> 下节课将学习 **MLP 从零搭建**：手写全连接层、激活函数、前向 / 反向完整实现。